In [11]:
import numpy as np
import pandas as pd
import csiread
import glob
import os
from scipy.stats import rice
import warnings

# Suppress scipy warnings for perfectly flat windows
warnings.filterwarnings('ignore')

def extract_all_wct_features(data_stream):
    Nc = 100
    Ns = 30
    Nf = 8
    
    k_series, std_series, par_series, cct_series = [], [], [], []
    
    # Step 1: CSI-Amplitude Window Loop
    for i in range(0, len(data_stream) - Nc, Ns):
        window = data_stream[i : i + Nc]
        
        # 1. Raw Amplitude STD
        std_series.append(np.std(window))
        
        # 2. Peak-to-Average Ratio (PAR)
        rms_sq = np.mean(np.square(window))
        par = (np.max(window)**2) / rms_sq if rms_sq > 0 else 0
        par_series.append(par)
        
        # 3. Channel Coherence Time (Autocorrelation Proxy)
        if np.var(window) == 0:
            cct_series.append(1.0)
        else:
            cct_series.append(np.corrcoef(window[:-1], window[1:])[0, 1])
            
        # 4. Rician K-Factor via MLE
        rms = np.sqrt(rms_sq) if rms_sq > 0 else 1
        v, loc, scale = rice.fit(window / rms, floc=0) 
        k_factor = (v**2) / (2 * (scale**2)) if scale > 0 else 100
        k_series.append(k_factor)
        
    # Step 2: Feature Window Stacking
    final_features = []
    threshold = np.median(k_series) 
    
    for j in range(0, len(k_series) - Nf):
        k_block = np.array(k_series[j : j + Nf])
        crossings = np.sum((k_block[:-1] >= threshold) & (k_block[1:] < threshold))
        time_below = np.sum(k_block < threshold)
        afd = time_below / crossings if crossings > 0 else time_below
        
        feature_vector = [
            np.mean(k_block),                     
            np.std(k_block),                      
            crossings,                            
            afd,                                  
            np.mean(std_series[j : j + Nf]),      
            np.mean(par_series[j : j + Nf]),      
            np.mean(cct_series[j : j + Nf])       
        ]
        final_features.append(feature_vector)
        
    return np.array(final_features)

print("Cell 1 Executed: WCT Physics Engine Loaded.")

Cell 1 Executed: WCT Physics Engine Loaded.


In [12]:
def build_mass_dataset(folder_path):
    all_features = []
    all_labels = []
    
    # Target the specific folder you created
    file_pattern = os.path.join(folder_path, '*.dat')
    file_list = glob.glob(file_pattern)
    
    print(f"Found {len(file_list)} target files. Initiating extraction...")
    
    for idx, file in enumerate(file_list):
        if idx > 0 and idx % 50 == 0:
            print(f"  -> Processed {idx}/{len(file_list)} files...")
            
        try:
            # Decode Intel 5300 packets
            csidata = csiread.Intel(file)
            csidata.read()
            
            # Extract Amplitude for Rx=0, Tx=0
            amplitude = np.abs(csidata.csi[:, :, 0, 0])
            
            # Extract features for a middle subcarrier (index 15)
            features = extract_all_wct_features(amplitude[:, 15])
            
            # Labeling Logic: Edges = 0 (Empty/Static), Middle = 1 (Moving)
            labels = np.zeros(len(features))
            if len(labels) > 20: 
                midpoint=len(labels)//2
                labels[midpoint - 4 : midpoint + 4] = 1
                
            all_features.append(features)
            all_labels.append(labels)
            
        except Exception as e:
            continue # Silently skip any corrupted files

    X_matrix = np.vstack(all_features)
    y_vector = np.concatenate(all_labels)
    
    return X_matrix, y_vector

print("Cell 2 Executed: Builder Logic Loaded.")

Cell 2 Executed: Builder Logic Loaded.


In [13]:
# Execute the builder on your specific folder
X, y = build_mass_dataset('Widar-dataset-office')

# Convert to a clean Pandas DataFrame
columns = ['Mean_K', 'Std_K', 'LCR', 'AFD', 'Mean_Raw_STD', 'Mean_PAR', 'Mean_CCT']
df = pd.DataFrame(X, columns=columns)
df['Occupancy_Label'] = y

# Save to CSV
csv_filename = 'WCT_Office_Dataset.csv'
df.to_csv(csv_filename, index=False)

print("\n--- DATASET GENERATION COMPLETE ---")
print(f"Total Physical Time-Steps Extracted: {df.shape[0]}")
print(f"Saved successfully to '{csv_filename}'")

Found 1499 target files. Initiating extraction...
  -> Processed 50/1499 files...
  -> Processed 100/1499 files...
  -> Processed 150/1499 files...
  -> Processed 200/1499 files...
  -> Processed 250/1499 files...
  -> Processed 300/1499 files...
  -> Processed 350/1499 files...
  -> Processed 400/1499 files...
  -> Processed 450/1499 files...
  -> Processed 500/1499 files...
  -> Processed 550/1499 files...
  -> Processed 600/1499 files...
  -> Processed 650/1499 files...
  -> Processed 700/1499 files...
  -> Processed 750/1499 files...
  -> Processed 800/1499 files...
  -> Processed 850/1499 files...
  -> Processed 900/1499 files...
  -> Processed 950/1499 files...
  -> Processed 1000/1499 files...
  -> Processed 1050/1499 files...
  -> Processed 1100/1499 files...
  -> Processed 1150/1499 files...
  -> Processed 1200/1499 files...
  -> Processed 1250/1499 files...
  -> Processed 1300/1499 files...
  -> Processed 1350/1499 files...
  -> Processed 1400/1499 files...
  -> Processed 145

In [20]:
# Display the first 5 rows of your engineered physics data
display(df.head())

# Check the balance of Empty (0) vs Occupied (1) data
print("\nClass Balance:")
print(df['Occupancy_Label'].value_counts())

,Mean_K,Std_K,LCR,AFD,Mean_Raw_STD,Mean_PAR,Mean_CCT,Occupancy_Label
0,10699.984500,3520.703679,0.0,8.0,1.604310,1.452008,0.704311,0.0
1,10640.567994,3597.862666,0.0,8.0,1.562684,1.460015,0.711148,0.0
2,10889.029770,3420.018081,0.0,8.0,1.508624,1.462732,0.716907,0.0
3,10864.016298,3414.754212,0.0,8.0,1.481748,1.437941,0.728245,0.0
4,12224.692908,4215.426809,0.0,7.0,1.420910,1.410039,0.724923,0.0



Class Balance:
Occupancy_Label
0.0    58972
1.0    11992
Name: count, dtype: int64


In [18]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print("--- LOADING WCT DATASET ---")
df = pd.read_csv('WCT_Office_Dataset.csv')

# 1. Clean the data (Sometimes math creates NaNs or Infs on perfectly flat Wi-Fi signals)
df = df.replace([np.inf, -np.inf], np.nan).dropna()

X = df.drop('Occupancy_Label', axis=1)
y = df['Occupancy_Label']

# 2. Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale the features (Crucial for gradient boosting)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Samples: {X_train_scaled.shape[0]}")
print(f"Testing Samples: {X_test_scaled.shape[0]}\n")

# ==========================================
# MODEL 1: XGBOOST
# ==========================================
print("--- Training XGBoost ---")
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, class_weight='balanced')

# Time the training
start_time = time.time()
xgb.fit(X_train_scaled, y_train)
xgb_train_time = time.time() - start_time

# Time the inference (Prediction)
start_time = time.time()
xgb_preds = xgb.predict(X_test_scaled)
xgb_infer_time = time.time() - start_time

xgb_acc = accuracy_score(y_test, xgb_preds)

# ==========================================
# MODEL 2: LIGHTGBM
# ==========================================
print("--- Training LightGBM ---")
lgbm = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, class_weight='balanced')

start_time = time.time()
lgbm.fit(X_train_scaled, y_train)
lgbm_train_time = time.time() - start_time

start_time = time.time()
lgbm_preds = lgbm.predict(X_test_scaled)
lgbm_infer_time = time.time() - start_time

lgbm_acc = accuracy_score(y_test, lgbm_preds)

# ==========================================
# THE RESULTS
# ==========================================
print("\n" + "="*50)
print("🏆 FINAL MODEL COMPARISON 🏆")
print("="*50)

print(f"{'Metric':<20} | {'XGBoost':<12} | {'LightGBM':<12}")
print("-" * 50)
print(f"{'Accuracy':<20} | {xgb_acc*100:>10.2f}% | {lgbm_acc*100:>10.2f}%")
print(f"{'Training Time':<20} | {xgb_train_time:>10.4f}s | {lgbm_train_time:>10.4f}s")
print(f"{'Total Inference Time':<20} | {xgb_infer_time:>10.4f}s | {lgbm_infer_time:>10.4f}s")

# Calculate Inference Time per Sample (in milliseconds)
xgb_ms_per_sample = (xgb_infer_time / len(y_test)) * 1000
lgbm_ms_per_sample = (lgbm_infer_time / len(y_test)) * 1000

print(f"{'Time per Prediction':<20} | {xgb_ms_per_sample:>10.4f}ms| {lgbm_ms_per_sample:>10.4f}ms")
print("="*50)

print("\nLightGBM Detailed Report:")
print(classification_report(y_test, lgbm_preds))

--- LOADING WCT DATASET ---
Training Samples: 56771
Testing Samples: 14193

--- Training XGBoost ---
--- Training LightGBM ---
[LightGBM] [Info] Number of positive: 9591, number of negative: 47180
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000639 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1294
[LightGBM] [Info] Number of data points in the train set: 56771, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Light

In [21]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# --- ADD THIS IMPORT ---
from imblearn.over_sampling import SMOTE

print("--- LOADING WCT DATASET ---")
df = pd.read_csv('WCT_Office_Dataset.csv')

df = df.replace([np.inf, -np.inf], np.nan).dropna()

X = df.drop('Occupancy_Label', axis=1)
y = df['Occupancy_Label']

# Split first (NEVER apply SMOTE before splitting, it causes data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Before SMOTE:")
print(y_train.value_counts())

# --- APPLY SMOTE ---
print("\nApplying SMOTE to balance the training set...")
smote = SMOTE(random_state=42)
X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

print("\nAfter SMOTE:")
print(y_train.value_counts())

# Now proceed with your XGBoost and LightGBM training using the new X_train_scaled and y_train
# (Remove the class_weight='balanced' from your models since the data is now physically balanced)

--- LOADING WCT DATASET ---
Before SMOTE:
Occupancy_Label
0.0    47180
1.0     9591
Name: count, dtype: int64

Applying SMOTE to balance the training set...

After SMOTE:
Occupancy_Label
0.0    47180
1.0    47180
Name: count, dtype: int64


In [24]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from scipy.stats import randint

print("--- INITIALIZING THE FINAL ML BAKE-OFF ---")

# 1. Data Prep (assuming 'df' is still in memory from your previous cell)
X = df.drop('Occupancy_Label', axis=1)
y = df['Occupancy_Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []

# ==========================================
# 1. Random Forest (The Paper's Baseline - Tuned)
# ==========================================
print("Training and Tuning Random Forest...")
rf_param_dist = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(10, 30),
    'min_samples_split': randint(2, 10)
}
rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(rf, param_distributions=rf_param_dist, n_iter=5, cv=3, scoring='accuracy', random_state=42, n_jobs=-1)

start_time = time.time()
rf_search.fit(X_train_scaled, y_train)
rf_train_time = time.time() - start_time

start_time = time.time()
rf_preds = rf_search.predict(X_test_scaled)
rf_infer_time = time.time() - start_time

rf_acc = accuracy_score(y_test, rf_preds)
results.append({
    "Model": "Random Forest (Tuned)", 
    "Accuracy": rf_acc, 
    "Train Time (s)": rf_train_time, 
    "Inference Time (s)": rf_infer_time,
    "Time per Pred (ms)": (rf_infer_time / len(y_test)) * 1000
})

# ==========================================
# 2. XGBoost (Our Champion)
# ==========================================
print("Training XGBoost...")
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, class_weight='balanced')

start_time = time.time()
xgb.fit(X_train_scaled, y_train)
xgb_train_time = time.time() - start_time

start_time = time.time()
xgb_preds = xgb.predict(X_test_scaled)
xgb_infer_time = time.time() - start_time

xgb_acc = accuracy_score(y_test, xgb_preds)
results.append({
    "Model": "XGBoost", 
    "Accuracy": xgb_acc, 
    "Train Time (s)": xgb_train_time, 
    "Inference Time (s)": xgb_infer_time,
    "Time per Pred (ms)": (xgb_infer_time / len(y_test)) * 1000
})

# ==========================================
# 3. LightGBM (The Edge-Computing Alternative)
# ==========================================
print("Training LightGBM...")
lgbm = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, class_weight='balanced', verbose=-1)

start_time = time.time()
lgbm.fit(X_train_scaled, y_train)
lgbm_train_time = time.time() - start_time

start_time = time.time()
lgbm_preds = lgbm.predict(X_test_scaled)
lgbm_infer_time = time.time() - start_time

lgbm_acc = accuracy_score(y_test, lgbm_preds)
results.append({
    "Model": "LightGBM", 
    "Accuracy": lgbm_acc, 
    "Train Time (s)": lgbm_train_time, 
    "Inference Time (s)": lgbm_infer_time,
    "Time per Pred (ms)": (lgbm_infer_time / len(y_test)) * 1000
})

# ==========================================
# THE FINAL BAKE-OFF TABLE
# ==========================================
print("\n" + "="*75)
print("🏆 FINAL ALGORITHMIC SHOWDOWN: IEEE BASELINE VS. GRADIENT BOOSTING 🏆")
print("="*75)

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)

# Formatting for clean output
print(f"{'Model':<25} | {'Accuracy':<10} | {'Train Time':<12} | {'Time/Pred (ms)':<15}")
print("-" * 75)
for index, row in results_df.iterrows():
    print(f"{row['Model']:<25} | {row['Accuracy']*100:>9.2f}% | {row['Train Time (s)']:>11.4f}s | {row['Time per Pred (ms)']:>13.4f}ms")
print("="*75)

print("\nBest Random Forest Parameters Found:")
print(rf_search.best_params_)

--- INITIALIZING THE FINAL ML BAKE-OFF ---
Training and Tuning Random Forest...
Training XGBoost...
Training LightGBM...

🏆 FINAL ALGORITHMIC SHOWDOWN: IEEE BASELINE VS. GRADIENT BOOSTING 🏆
Model                     | Accuracy   | Train Time   | Time/Pred (ms) 
---------------------------------------------------------------------------
Random Forest (Tuned)     |     83.58% |     22.9319s |        0.0033ms
XGBoost                   |     83.13% |      0.1020s |        0.0003ms
LightGBM                  |     59.87% |      0.0887s |        0.0000ms

Best Random Forest Parameters Found:
{'max_depth': 24, 'min_samples_split': 4, 'n_estimators': 171}


In [26]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from scipy.stats import randint
from imblearn.over_sampling import SMOTE

print("--- 1. DATA PREP & SMOTE BALANCING ---")
# Assume 'df' is loaded
X = df.drop('Occupancy_Label', axis=1)
y = df['Occupancy_Label']

# Split FIRST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Do NOT touch the test set size/balance

# Apply SMOTE to training data ONLY
smote = SMOTE(random_state=42)
X_train_scaled_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

results = []

# ==========================================
# 1. Random Forest (Removed class_weight)
# ==========================================
print("Training Random Forest...")
rf_param_dist = {'n_estimators': randint(100, 300), 'max_depth': randint(10, 30), 'min_samples_split': randint(2, 10)}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_search = RandomizedSearchCV(rf, param_distributions=rf_param_dist, n_iter=5, cv=3, scoring='f1', random_state=42, n_jobs=-1)

start_time = time.time()
rf_search.fit(X_train_scaled_smote, y_train_smote)
rf_train_time = time.time() - start_time
rf_preds = rf_search.predict(X_test_scaled)

results.append({
    "Model": "Random Forest", 
    "Accuracy": accuracy_score(y_test, rf_preds), 
    "F1-Score (Occupied)": f1_score(y_test, rf_preds), # The true success metric
    "Time/Pred (ms)": ((time.time() - start_time) / len(y_test)) * 1000
})

# ==========================================
# 2. XGBoost (Removed class_weight)
# ==========================================
print("Training XGBoost...")
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
start_time = time.time()
xgb.fit(X_train_scaled_smote, y_train_smote)
xgb_preds = xgb.predict(X_test_scaled)

results.append({
    "Model": "XGBoost", 
    "Accuracy": accuracy_score(y_test, xgb_preds), 
    "F1-Score (Occupied)": f1_score(y_test, xgb_preds),
    "Time/Pred (ms)": ((time.time() - start_time) / len(y_test)) * 1000
})

# ==========================================
# 3. LightGBM (Removed class_weight)
# ==========================================
print("Training LightGBM...")
lgbm = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1)
start_time = time.time()
lgbm.fit(X_train_scaled_smote, y_train_smote)
lgbm_preds = lgbm.predict(X_test_scaled)

results.append({
    "Model": "LightGBM", 
    "Accuracy": accuracy_score(y_test, lgbm_preds), 
    "F1-Score (Occupied)": f1_score(y_test, lgbm_preds),
    "Time/Pred (ms)": ((time.time() - start_time) / len(y_test)) * 1000
})

# ==========================================
# THE RESULTS
# ==========================================
print("\n" + "="*85)
print("🏆 FINAL ALGORITHMIC SHOWDOWN (TRAINED ON BALANCED SMOTE DATA) 🏆")
print("="*85)

results_df = pd.DataFrame(results).sort_values(by="F1-Score (Occupied)", ascending=False)

print(f"{'Model':<20} | {'Accuracy':<10} | {'F1-Score':<10} | {'Time/Pred (ms)':<15}")
print("-" * 85)
for index, row in results_df.iterrows():
    print(f"{row['Model']:<20} | {row['Accuracy']*100:>9.2f}% | {row['F1-Score (Occupied)']*100:>9.2f}% | {row['Time/Pred (ms)']:>13.4f}ms")
print("="*85)

print("\n--- DETAILED XGBOOST REPORT ---")
print(classification_report(y_test, xgb_preds))

--- 1. DATA PREP & SMOTE BALANCING ---
Training Random Forest...
Training XGBoost...
Training LightGBM...

🏆 FINAL ALGORITHMIC SHOWDOWN (TRAINED ON BALANCED SMOTE DATA) 🏆
Model                | Accuracy   | F1-Score   | Time/Pred (ms) 
-------------------------------------------------------------------------------------
Random Forest        |     79.06% |     36.39% |        2.8298ms
LightGBM             |     67.98% |     28.55% |        0.0102ms
XGBoost              |     67.02% |     28.55% |        0.0119ms

--- DETAILED XGBOOST REPORT ---
              precision    recall  f1-score   support

         0.0       0.85      0.73      0.79     11792
         1.0       0.23      0.39      0.29      2401

    accuracy                           0.67     14193
   macro avg       0.54      0.56      0.54     14193
weighted avg       0.75      0.67      0.70     14193

